In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/isic-flagship-project")
os.chdir(PROJECT_ROOT)

print("Working in:", PROJECT_ROOT)

Working in: /content/drive/MyDrive/isic-flagship-project


In [ ]:
%%writefile /content/drive/MyDrive/isic-flagship-project/src/api/main.py
"""
Cloud Run entry point for the Copilot Studio support API.
"""

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware



app = FastAPI(
    title="ISIC Skin Lesion Platform Support API",
    version="1.0.0",
    description="Lightweight API used by the Copilot Studio support agent.",
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.get("/")
async def root():
    return {
        "status": "ok",
        "service": "ISIC Copilot Support API",
    }


@app.get("/api/v1/health")
async def health():
    return {
        "status": "ok",
        "service": "copilot-support-api",
    }


app.include_router(
    copilot_router,
    prefix="/api/v1",
    tags=["copilot-studio"],
)

Overwriting /content/drive/MyDrive/isic-flagship-project/src/api/main.py


In [ ]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

from src.api.routes import health_router, prediction_router

app = FastAPI(
    title="ISIC 2024 Skin Cancer Detection API",
    version="1.0.0",
    description="ISIC Skin Lesion Platform API with image prediction endpoints",
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/")
async def root():
    return {
        "status": "ok",
        "service": "ISIC Skin Lesion Platform API",
        "docs_url": "/docs",
    }

app.include_router(health_router, prefix="/api/v1", tags=["health"])
app.include_router(prediction_router, prefix="/api/v1", tags=["prediction"])


In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/isic-flagship-project")
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))



routes = sorted(route.path for route in app.routes)

print("App import OK")
print("Routes:")
for route in routes:
    print("-", route)

assert "/api/v1/health" in routes
assert "/api/v1/predict" in routes

assert "/api/v1/agent/health" not in routes
assert "/api/v1/agent/support" not in routes


App import OK
Routes:
- /
- /api/v1/health
- /api/v1/predict
- /docs
- /docs/oauth2-redirect
- /openapi.json
- /redoc


In [ ]:
%%writefile /content/drive/MyDrive/isic-flagship-project/Dockerfile
FROM python:3.11-slim

WORKDIR /app

ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1

RUN apt-get update && apt-get install -y \
    git \
    curl \
    libgl1 \
    libglib2.0-0 \
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .

RUN pip install --upgrade pip

RUN pip install --no-cache-dir \
    torch==2.3.1 \
    torchvision==0.18.1 \
    --index-url https://download.pytorch.org/whl/cpu

RUN pip install --no-cache-dir -r requirements.txt

COPY src/ ./src/
COPY .env.example .env

EXPOSE 8080

CMD exec uvicorn src.api.main:app --host 0.0.0.0 --port ${PORT:-8080}

Overwriting /content/drive/MyDrive/isic-flagship-project/Dockerfile


In [ ]:
%%writefile /content/drive/MyDrive/isic-flagship-project/requirements.txt
fastapi==0.115.2
uvicorn[standard]==0.32.0
pydantic==2.9.2
pydantic-settings==2.6.1
sqlalchemy==2.0.35
alembic==1.13.2
aiosqlite==0.20.0
python-dotenv==1.0.1
pandas==2.2.2
numpy==1.26.4
pillow==10.4.0
timm==1.0.11
lightgbm==4.5.0
catboost==1.2.7
xgboost==2.1.1
scikit-learn==1.5.2
langchain==0.3.4
langchain-community==0.3.3
langchain-text-splitters==0.3.0
faiss-cpu==1.8.0.post1
sentence-transformers==3.1.1
python-multipart==0.0.9
httpx==0.27.2
structlog==24.4.0

Overwriting /content/drive/MyDrive/isic-flagship-project/requirements.txt


In [ ]:
%%writefile /content/drive/MyDrive/isic-flagship-project/.dockerignore
__pycache__/
*.pyc
*.pyo
*.pyd
.Python
env/
venv/
.venv/
.git/
.gitignore
.ipynb_checkpoints/
*.ipynb
logs/

Overwriting /content/drive/MyDrive/isic-flagship-project/.dockerignore


In [ ]:
%%writefile /content/drive/MyDrive/isic-flagship-project/src/api/main.py
"""
Main FastAPI application.
"""

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

from src.core.config import get_settings
from src.api.routes import health_router, prediction_router
from src.api.copilot_routes import copilot_router


settings = get_settings()

tags_metadata = [
    {
        "name": "prediction",
        "description": "Upload a skin lesion image and receive a malignant-risk prediction.",
    },
    {
        "name": "health",
        "description": "Health check endpoints.",
    },
    {
        "name": "copilot-studio",
        "description": "Support endpoints used by the Copilot Studio custom connector.",
    },
    {
        "name": "general",
        "description": "General API information.",
    },
]

app = FastAPI(
    title="ISIC Skin Lesion Platform API",
    version=settings.API_VERSION,
    description=(
        "ISIC Skin Lesion Detection API with image prediction "
        "and Copilot Studio support endpoints."
    ),
    openapi_tags=tags_metadata,
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.get("/", tags=["general"])
async def root():
    return {
        "status": "ok",
        "service": "ISIC Skin Lesion Platform API",
        "docs_url": "/docs",
        "openapi_url": "/openapi.json",
    }


app.include_router(
    prediction_router,
    prefix="/api/v1",
    tags=["prediction"],
)

app.include_router(
    health_router,
    prefix="/api/v1",
    tags=["health"],
)

app.include_router(
    copilot_router,
    prefix="/api/v1",
    tags=["copilot-studio"],
)

Overwriting /content/drive/MyDrive/isic-flagship-project/src/api/main.py


In [ ]:
%%writefile /content/drive/MyDrive/isic-flagship-project/src/api/routes.py
"""
API routes for skin lesion prediction.
"""

from datetime import datetime

from fastapi import APIRouter, UploadFile, File, HTTPException

from src.core.config import get_settings
from src.schemas.prediction import PredictionResponse, HealthResponse
from src.services.prediction_service import PredictionService


settings = get_settings()

health_router = APIRouter()
prediction_router = APIRouter()

prediction_service = PredictionService()


@health_router.get("/health", response_model=HealthResponse)
async def health_check():
    return HealthResponse(
        status="healthy",
        model_version=settings.MODEL_VERSION,
        api_version=settings.API_VERSION,
    )


@prediction_router.post(
    "/predict",
    response_model=PredictionResponse,
    summary="Expand the block to predict skin lesion risk",
    description="""
Upload a skin lesion image and receive a malignant-risk prediction.

If the probability is in the review boundary range, the backend may trigger
a human-review workflow depending on the active PredictionService implementation.
""",
)
async def predict_image(file: UploadFile = File(...)):
    if not file.content_type or not file.content_type.startswith("image/"):
        raise HTTPException(status_code=400, detail="File must be an image")

    result = await prediction_service.predict(file)

    probability = float(result["probability"])

    return PredictionResponse(
        isic_id=file.filename or "uploaded_image",
        probability=probability,
        prediction=result["prediction"],
        model_version=result.get("model_version", settings.MODEL_VERSION),
        timestamp=datetime.utcnow(),
        review_triggered=result.get(
            "review_triggered",
            0.45 <= probability <= 0.55,
        ),
    )

Overwriting /content/drive/MyDrive/isic-flagship-project/src/api/routes.py


In [ ]:
from google.colab import auth
auth.authenticate_user()

!gcloud auth list
!gcloud config get-value account

      Credentialed Accounts
ACTIVE  ACCOUNT
*       bo22454220a.stu@gmail.com

To set the active account, run:
    $ gcloud config set account `ACCOUNT`



To take a quick anonymous survey, run:
  $ gcloud survey

bo22454220a.stu@gmail.com


In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "isic-flagship-project-a"
REGION = "europe-west2"
SERVICE_NAME = "isic-api"

!gcloud config set project {PROJECT_ID}
!gcloud auth list
!gcloud config list
!gcloud projects describe {PROJECT_ID}

Updated property [core/project].
      Credentialed Accounts
ACTIVE  ACCOUNT
*       bo22454220a.stu@gmail.com

To set the active account, run:
    $ gcloud config set account `ACCOUNT`

[component_manager]
disable_update_check = True
[core]
account = bo22454220a.stu@gmail.com
project = isic-flagship-project-a
universe_domain = googleapis.com

Your active configuration is: [default]
createTime: '2026-07-02T20:59:01.864Z'
lifecycleState: ACTIVE
name: ISIC Flagship Project A
projectId: isic-flagship-project-a
projectNumber: '616904025812'


In [ ]:
PROJECT_NUMBER = !gcloud projects describe {PROJECT_ID} --format="value(projectNumber)"
PROJECT_NUMBER = PROJECT_NUMBER[0]

SERVICE_ACCOUNT = f"{PROJECT_NUMBER}-compute@developer.gserviceaccount.com"

print("Project number:", PROJECT_NUMBER)
print("Cloud Run service account:", SERVICE_ACCOUNT)

Project number: 616904025812
Cloud Run service account: 616904025812-compute@developer.gserviceaccount.com


In [ ]:
PROJECT_ID = "isic-flagship-project-a"
REGION = "europe-west2"
SERVICE_NAME = "isic-api"

!gcloud config set account bo22454220a.stu@gmail.com
!gcloud config set project {PROJECT_ID}

!gcloud services enable \
  run.googleapis.com \
  cloudbuild.googleapis.com \
  artifactregistry.googleapis.com \
  iam.googleapis.com \
  secretmanager.googleapis.com \
  --project={PROJECT_ID}

Updated property [core/account].
Updated property [core/project].
Operation "operations/acat.p2-616904025812-a2b1a3f1-c75d-44ed-9a82-8bbae872e3a0" finished successfully.


In [ ]:
!gcloud secrets add-iam-policy-binding power-automate-url \
  --member="serviceAccount:{SERVICE_ACCOUNT}" \
  --role="roles/secretmanager.secretAccessor" \
  --project={PROJECT_ID}

Updated IAM policy for secret [power-automate-url].
bindings:
- members:
  - serviceAccount:616904025812-compute@developer.gserviceaccount.com
  role: roles/secretmanager.secretAccessor
etag: BwZWQZu8v7Y=
version: 1


In [ ]:
%cd /content/drive/MyDrive/isic-flagship-project

!gcloud run deploy {SERVICE_NAME} \
  --source . \
  --region {REGION} \
  --allow-unauthenticated \
  --memory 4Gi \
  --cpu 1 \
  --min-instances 0 \
  --max-instances 1 \
  --timeout 300 \
  --port 8080 \
  --cpu-boost \
  --set-secrets POWER_AUTOMATE_URL=power-automate-url:latest \
  --project {PROJECT_ID}

/content
Building using Dockerfile and deploying container to Cloud Run service [isic-api] in project [isic-flagship-project-a] region [europe-west2]
Service [isic-api] revision [isic-api-00008-4x4] has been deployed and is serving 100 percent of traffic.
Service URL: https://isic-api-616904025812.europe-west2.run.app


In [ ]:
import requests

CLOUD_RUN_URL = "https://isic-api-616904025812.europe-west2.run.app"

print("Cloud Run URL:", CLOUD_RUN_URL)

health = requests.get(f"{CLOUD_RUN_URL}/api/v1/health", timeout=30)
print("Health:", health.status_code)
print(health.json())

openapi = requests.get(f"{CLOUD_RUN_URL}/openapi.json", timeout=30).json()
paths = sorted(openapi.get("paths", {}).keys())

print("OpenAPI paths:")
for path in paths:
    print("-", path)

assert "/api/v1/predict" in paths
assert "/api/v1/health" in paths

print("Cloud Run deployment check passed.")

Cloud Run URL: https://isic-api-616904025812.europe-west2.run.app
Health: 200
{'status': 'healthy', 'model_version': '2024-ensemble-v1', 'api_version': 'v1'}
OpenAPI paths:
- /
- /api/v1/agent/health
- /api/v1/agent/support
- /api/v1/health
- /api/v1/predict
Cloud Run deployment check passed.
